# Training a Denoising Diffusion Probabilistic Model (DDPM) <BR>with `diffusers` Library

Source:<P>

https://github.com/huggingface/diffusers/tree/main/examples

Adapted:<P>

Antonio Esteves @ UMinho, April 2025<P>

---

In recent years, it became clear that diffusion models have are the state-of-the-art image generation models. In this notebook, we will use Hugging Face's [diffusers](https://github.com/huggingface/diffusers) library to train a simple DDPM model.

## Installing the dependencies

This notebook leverages the [`datasets`](https://huggingface.co/docs/datasets/index) library to load and preprocess image datasets and the [`accelerate`](https://huggingface.co/docs/accelerate/index) library to simplify training on any number of GPUs, with features like automatic gradient accumulation and tensorboard logging.

Let us install them if necessary.

In [ ]:
# !pip install diffusers["torch"] transformers
# !pip install accelerate
# !pip install datasets

If running this notebook locally, make sure the python environment has the last version of those libraries installed.

To be able to share the model with the community, there are a few more steps to follow.

First we have to store our authentication token from the Hugging Face website (sign up [here](https://huggingface.co/join) if have not already registered), then execute the following cell and input the token.

In [ ]:
# !huggingface-cli login

In [ ]:
!huggingface-cli whoami

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


Then, we need to install `git-lfs` to upload our model checkpoints to Hugging Face (HF) hub.

In [ ]:
# !sudo dnf install git-lfs
# !git config --global credential.helper store

## Configuration

First, we define a configuration with all the necessary training hyperparameters. Here we choose reasonable default values for hyperparameters such as `num_epochs`, `learning_rate`, `lr_warmup_steps`.

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    image_size                  = 128    # the generated image resolution
    train_batch_size            = 16     # the training batch size
    eval_batch_size             = 16     # how many images to sample during evaluation
    num_epochs                  = 50     # number of epochs
    gradient_accumulation_steps = 1      # number of steps used to accumulate gradients
    learning_rate               = 0.0001 # the learning rate 
    lr_warmup_steps             = 500    # number of steps used during learning rate warmup
    save_image_epochs           = 10     # interval between saving successive sampled images
    save_model_epochs           = 10     # interval between saving uccessive model checkpoints
    mixed_precision             = 'fp16' # 'no' for float32, 'fp16' for automatic mixed precision
    output_dir                  = 'ddpm-butterflies-128'  # the model name (locally and on the HF Hub)
    push_to_hub                 = True   # whether to upload the saved model to the HF Hub (True) or not (False)
    hub_private_repo            = False  # make the HP repo private (True) or public (False)
    overwrite_output_dir        = True   # overwrite the old model when re-running the notebook (True) or not (False)
    seed                        = 0
    hub_model_id                = None
    hub_token                   = 'ouur_token_here' # our HF tokem

config = TrainingConfig()

## Loading the dataset

We will use the [`datasets`](https://github.com/huggingface/datasets) library to download the dataset.

In this case, the [Butterflies dataset](https://huggingface.co/datasets/huggan/smithsonian_butterflies_subset) is hosted remotely, but we can load a local [ImageFolder](https://huggingface.co/docs/datasets/v2.0.0/en/image_process#imagefolder) as shown in the comments below.

In [ ]:
from datasets import load_dataset

config.dataset = "huggan/smithsonian_butterflies_subset"
dataset        = load_dataset(config.dataset, split="train")

# We can just load images from a local folder:
# config.dataset = "imagefolder"
# dataset        = load_dataset(config.dataset, data_dir="path/to/folder")

The dataset contains several extra `features` (columns), but the one that we are interested in is `image`.

In [ ]:
dataset

Since the [`Image`](https://huggingface.co/docs/datasets/image_process#image-datasets) feature loads the images with PIL, we can easily look at a few examples.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for i, image in enumerate(dataset[:4]["image"]):
    axs[i].imshow(image)
    axs[i].set_axis_off()
fig.show()

The images in the dataset are all different, so we need to preprocess them first:
* `Resize` makes the images conform to a square resolution of `config.image_size`
* `RandomHorizontalFlip` augments the dataset by randomly mirroring the images.
* `Normalize` is important to rescale the pixel values into a `[-1, 1]` range, which our model will expect.

In [ ]:
from torchvision import transforms

preprocess = transforms.Compose(
    [
        transforms.Resize((config.image_size, config.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

`datasets` offer a handy `set_transform` method to apply the image transformations on the fly during training.

In [ ]:
def transform(examples):
    images = [preprocess(image.convert("RGB")) for image in examples["image"]]
    return {"images": images}

dataset.set_transform(transform)

Let us see what the transformed imaes look like.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for i, image in enumerate(dataset[:4]["images"]):
    axs[i].imshow(image.permute(1, 2, 0).numpy() / 2 + 0.5)
    axs[i].set_axis_off()
fig.show()

Now that all our images have the same size and are converted to tensors, we can create the dataloader we will use for training.

In [ ]:
import torch

train_dataloader = torch.utils.data.DataLoader(
  dataset,
  batch_size = config.train_batch_size,
  shuffle    = True,
)

## Defining the denoising diffusion probabilistic model (DDPM)

Here we define our diffusion model. Diffusion models are neural networks that are trained to predict slightly less noisy images from a noisy input. At inference, they can be used to iteratively transform a random noise to generate an image:

<p align="center">
    <img src="https://user-images.githubusercontent.com/10695622/174349667-04e9e485-793b-429a-affe-096e8199ad5b.png" width="800"/>
    <br>
    <em> Figure from DDPM paper (https://arxiv.org/abs/2006.11239). </em>
<p>

The model corresponds to the $p_{\theta}(x_{t-1}|x_{t})$, which given an input image predicts a slightly less noisy image. The interesting part is that it is really easy to add some noise to an image, so the training can happen in a semi-supervised fashion as follows:
1. Take an image from the training set.
2. Apply to it some random noise $t$ times, this will give us $x_{t-1}$ and $x_{t}$ in the figure above.
3. Apply the noisy image into the model along with the value of $t$.
4. Compute a loss from the output of the model and the noised image $x_{t-1}$.

Then, we can apply gradient descent and repeat this process multiple times.

Most diffusion models use a neural network that is variant of the [U-Net](https://arxiv.org/abs/1505.04597) and that is what we will use here.

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/unet-model.png)

A summary of the U-Net architecture is:
- the model passes the input image through several downsampling ResNet blocks, where each block halves the image size by 2.
- then, the feature maps go through the same number of upsampling ResNet blocks.
- there are skip connections to combine features from the downample path to the corresponsding features in the upsample path.

A key feature of this model is that it predicts images of the same size as the input, which is exactly what we need here.

Diffusers provides us a handy `UNet2DModel` class which creates the desired architecture in PyTorch. So, let us create a U-net for our desired image size. Note that `down_block_types` correspond to the downsampling blocks (green on the diagram above), and `up_block_types` are the upsampling blocks (red on the diagram).


In [ ]:
from diffusers import UNet2DModel

model = UNet2DModel(
    sample_size        = config.image_size,  # the target image resolution
    in_channels        = 3,  # the number of input channels, 3 for RGB images
    out_channels       = 3,  # the number of output channels
    layers_per_block   = 2,  # how many ResNet layers to use per U-Net block
    block_out_channels = (128, 128, 256, 256, 512, 512),  # the number of output channes for each U-Net block
    down_block_types   = (
        "DownBlock2D",  # a regular ResNet downsampling block
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",  # a ResNet downsampling block with self-attention
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",      # a regular ResNet upsampling block
        "AttnUpBlock2D",  # a ResNet upsampling block with self-attention
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
      ),
)

Let us get a sample image from our dataset and pass it into our model. We just need to add a batch dimension.

In [ ]:
sample_image = dataset[0]['images'].unsqueeze(0)
print('Input shape:', sample_image.shape)

And let us check the output is a tensor of the same exact shape.

In [ ]:
print('Output shape:', model(sample_image, timestep=0)["sample"].shape)

Our model takes in the noisy image and also the current timestep, as we saw before in the training overview. The timestep information is converted for the model using a sinusoidal positional embedding, similar to what the transformer model often does.

Now that we have our model, we need an entity to add noise to an image. This is done by the scheduler in the `diffusers` library.

## Defining the noise scheduler

Depending on the diffusion algorithm we want to use, the way images are noised is slightly different. That is why `diffusers` contains different schedulers, each one defines the algorithm-specific diffusion steps. Here we are going to use the `DDPMScheduler`, which corresponds to the training algorithm proposed in [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239).

In [ ]:
from diffusers import DDPMScheduler

noise_scheduler = DDPMScheduler(
  num_train_timesteps=1000,
)

Let us see how this noise scheduler works: it takes a batch of images from the trainng set, here we will reuse the batch of one image `sample_image` form before, a batch of random noise of the same shape, and the timesteps for each image, which correspond to the number of times we want to apply noise to each image.

In [ ]:
import torch
from   PIL   import Image

noise       = torch.randn(sample_image.shape)
timesteps   = torch.LongTensor([50])
noisy_image = noise_scheduler.add_noise(sample_image, noise, timesteps)

Image.fromarray(
  (
    (noisy_image.permute(0, 2, 3, 1) + 1.0) * 127.5
  ).type(torch.uint8).numpy()[0])

In the DDPM algorithm, the training objective of the model is then to be able to predict the noise we used in `noise_scheduler.add_noise`, so the loss at this step would be the mean squared error (MSE) between added and predicted noises.

In [ ]:
import torch.nn.functional as F

noise_pred = model(noisy_image, timesteps)["sample"]
loss       = F.mse_loss(noise_pred, noise)

## Setup U-Net training

We have all we need to be able to train the model. We are going to use the AdamW optimizer.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

 And a cosine learning rate schedule to dinamycally adjust the learning rate.

In [ ]:
from diffusers.optimization import get_cosine_schedule_with_warmup

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer          = optimizer,
    num_warmup_steps   = config.lr_warmup_steps,
    num_training_steps = (len(train_dataloader) * config.num_epochs),
)

To evaluate our model, we use the `DDPMPipeline` which is an easy way to perform end-to-end inference. We will use this pipeline to generate a batch of sampled images and save it as a grid to the disk.

In [ ]:
from   diffusers import DDPMPipeline
import math

def make_grid(images, rows, cols):
    w, h = images[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    for i, image in enumerate(images):
        grid.paste(image, box=(i%cols*w, i//cols*h))
    return grid

def evaluate(config, epoch, pipeline):
    # Sample some images from random noise (this is the backward diffusion process).
    # The default pipeline output type is 'List[PIL.Image]'
    images = pipeline(
        batch_size = config.eval_batch_size,
        generator  = torch.manual_seed(config.seed),
    ).images

    # Make a grid with the images
    image_grid = make_grid(images, rows=4, cols=4)

    # Save the images to file
    test_dir = os.path.join(config.output_dir, "samples")
    os.makedirs(test_dir, exist_ok=True)
    image_grid.save(f"{test_dir}/{epoch:04d}.png")

With this in end, we can group all together and write our training function. This just wraps the training step we saw in the previous section in a loop, using `accelerate` for logging, gradient accumulation, mixed precision training, and multi-GPU training.

In [ ]:
import os
from   accelerate          import Accelerator
from   tqdm.auto           import tqdm
from   pathlib             import Path
from   typing              import Optional
from   huggingface_hub     import HfFolder, Repository, whoami
from   huggingface_hub     import create_repo, hf_hub_download

# Check if the repository already exists before attempting to create it.
# If it exists, download the model instead of creating a new repository.
try:
    create_repo("ajesteves/ddpm-butterflies-128", exist_ok=False) # Set exist_ok=False to raise an error if the repo already exists
except Exception as e:
    if "409" in str(e):  # Check if the error is a '409 Conflict error'
        print("Repository already exists.")
    else:
        raise e  # Raise the original error if it is not a 409 error

def get_full_repo_name(model_id: str, organization: Optional[str] = None, token: Optional[str] = None):
    if token is None:
        token = HfFolder.get_token()
    if organization is None:
        username = whoami(token)["name"]
        return f"{username}/{model_id}"
    else:
        return f"{organization}/{model_id}"

def train_loop(config, model, noise_scheduler, optimizer, train_dataloader, lr_scheduler):
    # Initialize accelerator and tensorboard logging
    accelerator = Accelerator(
        mixed_precision             = config.mixed_precision,
        gradient_accumulation_steps = config.gradient_accumulation_steps,
    )
    if accelerator.is_main_process:

      if config.push_to_hub:
          if config.hub_model_id is None:
              repo_name = get_full_repo_name(
                  Path(config.output_dir).name,
                  token = config.hub_token,
              )
          else:
              repo_name = config.hub_model_id
          repo = Repository(config.output_dir, clone_from=repo_name)

          with open(os.path.join(config.output_dir, ".gitignore"), "w+") as gitignore:
              if "step_*" not in gitignore:
                  gitignore.write("step_*\n")
              if "epoch_*" not in gitignore:
                  gitignore.write("epoch_*\n")
      elif config.output_dir is not None:
          os.makedirs(config.output_dir, exist_ok=True)

    # Prepare everything
    model, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
      model,
      optimizer,
      train_dataloader,
      lr_scheduler
    )
    global_step = 0

    # Now we train the model
    for epoch in range(config.num_epochs):
        progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")

        for step, batch in enumerate(train_dataloader):
            clean_images = batch['images']
            # Sample noise to add to the images
            noise = torch.randn(clean_images.shape).to(clean_images.device)
            bs    = clean_images.shape[0]

            # Sample a random timestep for each image
            timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (bs,), device=clean_images.device).long()

            # Add noise to the clean images according to the noise magnitude at each timestep
            # (this is the forward diffusion process)
            noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

            with accelerator.accumulate(model):
                # Predict the noise to be removed
                noise_pred = model(noisy_images, timesteps)["sample"]
                loss       = F.mse_loss(noise_pred, noise)
                accelerator.backward(loss)

                accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0], "step": global_step}
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)
            global_step += 1

        # After each epoch you optionally sample some demo images with 'evaluate' and save the model
        if accelerator.is_main_process:
            pipeline = DDPMPipeline(unet=accelerator.unwrap_model(model), scheduler=noise_scheduler)

            if (epoch + 1) % config.save_image_epochs == 0 or epoch == config.num_epochs - 1:
                evaluate(config, epoch, pipeline)

            if (epoch + 1) % config.save_model_epochs == 0 or epoch == config.num_epochs - 1:
                if config.push_to_hub:
                    commit_msg = f"Epoch {epoch}"
                    pipeline.push_to_hub(repo_name, commit_message=commit_msg)
                else:
                    pipeline.save_pretrained(config.output_dir)

## Train the U-Net

Let us launch the training (including multi-GPU training) from the notebook using the `accelerate` `notebook_launcher` function.

In [ ]:
from accelerate import notebook_launcher

args = (config, model, noise_scheduler, optimizer, train_dataloader, lr_scheduler)

notebook_launcher(train_loop, args, num_processes=1)

Let us have a look at the final image grid produced by the trained diffusion model.

In [ ]:
import glob

sample_images = sorted(glob.glob(f"{config.output_dir}/samples/*.png"))
Image.open(sample_images[-1])

Not bad, but there is room for improvement by adjusting the hyperparameters, the model definition and image augmentations.

If we chose to upload the model to the Hugging Face Hub, its repository should now look like this:
https://huggingface.co/ajesteves/ddpm-butterflies-128